# 02 · Classical models

Every classical candidate trained on the same `Data/train.csv` and scored on the same `Data/test.csv`. Run notebook 01 first. Runs on a laptop CPU in about 10 minutes.

Two improvements over the first version, both measured on this data:
- **Input = review title + text.** The title a reviewer writes ("Great little speaker", "Disappointed") is a strong signal. Amazon's automatic titles like "Five Stars" were blanked in notebook 01 because they leak the rating.
- **Cutoff tuning.** Each model's per-class cutoffs are adjusted to maximise macro-F1, using 5-fold cross-validation on the training set only (the test set is never used for tuning).

All results are written to `Output/results/` so notebook 04 can compare them with the transformers.

In [1]:
from pathlib import Path
# Works whether the notebook runs from the Notebooks folder (VS Code default) or the project root
ROOT = Path.cwd().parent if Path.cwd().name == 'Notebooks' else Path.cwd()
DATA, OUTPUT = ROOT / 'Data', ROOT / 'Output'
RESULTS = OUTPUT / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)

import json, time
import numpy as np, pandas as pd
from sklearn.pipeline import make_pipeline, make_union
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import classification_report, f1_score

LABELS = ['not satisfied', 'neutral', 'satisfied']
train = pd.read_csv(DATA / 'train.csv').dropna(subset=['text'])
test = pd.read_csv(DATA / 'test.csv').dropna(subset=['text'])

def save_result(name, family, y_true, y_pred, extra=None):
    """One JSON per model in Output/results, read by notebook 04."""
    r = classification_report(y_true, y_pred, labels=LABELS, output_dict=True, zero_division=0)
    out = {'model': name, 'family': family, 'macro_f1': r['macro avg']['f1-score'], 'accuracy': r['accuracy'],
           **{f'recall_{l}': r[l]['recall'] for l in LABELS}, **{f'f1_{l}': r[l]['f1-score'] for l in LABELS}, **(extra or {})}
    json.dump(out, open(RESULTS / f'{name}.json', 'w'), indent=2)
    return out

def model_input(df):
    """Review title (when the reviewer wrote one) + text. Notebook 01 already blanked Amazon's automatic 'Five Stars' titles."""
    if 'title' not in df:
        raise ValueError('Data/train.csv has no title column: re-run notebook 01 first (same split, just adds titles).')
    title, text = df['title'].fillna('').astype(str).str.strip(), df['text'].astype(str)
    return np.where(title != '', title + ' . ' + text, text).tolist()

def tune_offsets(scores, y):
    """Per-class cutoff adjustment: add an offset to each class score, chosen to maximise macro-F1 on held-out data.
    scores: n x 3 array in LABELS order (log-probabilities or decision scores); y: true label indices."""
    grid = np.round(np.arange(-1.5, 1.51, 0.1), 2)
    best_f1, best = f1_score(y, scores.argmax(1), average='macro'), np.zeros(3)
    for a in grid:
        for b in grid:
            off = np.array([a, b, 0.0])   # not satisfied, neutral; satisfied is the reference
            f = f1_score(y, (scores + off).argmax(1), average='macro')
            if f > best_f1 + 1e-9:
                best_f1, best = f, off
    return best

X_train, X_test = model_input(train), model_input(test)
y_train = train.label.map({l: i for i, l in enumerate(LABELS)}).values

In [2]:
word = lambda: TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
word_char = lambda: make_union(
    TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=200_000),
    TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), min_df=3, sublinear_tf=True, max_features=300_000))

CANDIDATES = {
    'multinomial_nb':          make_pipeline(word(), MultinomialNB(alpha=0.1)),
    'complement_nb':           make_pipeline(word(), ComplementNB(alpha=0.3)),
    'logreg_unweighted':       make_pipeline(word(), LogisticRegression(max_iter=2000)),
    'logreg_balanced':         make_pipeline(word(), LogisticRegression(max_iter=2000, class_weight='balanced')),
    'sgd_hinge_balanced':      make_pipeline(word(), SGDClassifier(alpha=5e-5, class_weight='balanced', random_state=42)),
    'wordchar_logreg_balanced':make_pipeline(word_char(), LogisticRegression(max_iter=3000, class_weight='balanced')),
    'wordchar_svm_balanced':   make_pipeline(word_char(), LinearSVC(C=0.1, class_weight='balanced')),
}

def scores_of(pipe, X, method):
    s = getattr(pipe, method)(X)
    s = np.log(np.clip(s, 1e-9, 1)) if method == 'predict_proba' else s
    order = [list(pipe.classes_).index(l) for l in LABELS]   # sklearn sorts classes alphabetically
    return s[:, order]

cv = StratifiedKFold(5, shuffle=True, random_state=42)
rows = []
for name, pipe in CANDIDATES.items():
    t = time.time()
    method = 'predict_proba' if hasattr(pipe, 'predict_proba') else 'decision_function'
    oof = cross_val_predict(pipe, X_train, train.label, cv=cv, method=method)   # out-of-fold scores, train only
    oof = np.log(np.clip(oof, 1e-9, 1)) if method == 'predict_proba' else oof   # cross_val_predict orders by sorted class
    oof = oof[:, [sorted(LABELS).index(l) for l in LABELS]]
    offsets = tune_offsets(oof, y_train)
    pipe.fit(X_train, train.label)
    s = scores_of(pipe, X_test, method)
    untuned = f1_score(test.label, [LABELS[i] for i in s.argmax(1)], average='macro')
    pred = [LABELS[i] for i in (s + offsets).argmax(1)]
    rows.append(save_result(name, 'classical', test.label, pred,
                            {'input': 'title+text', 'macro_f1_untuned': untuned, 'offsets': offsets.tolist(),
                             'train_seconds': round(time.time() - t)}))
    print(f'{name:26s} macro-F1 {untuned:.3f} -> {rows[-1]["macro_f1"]:.3f} after cutoff tuning  ({time.time() - t:.0f}s)')

multinomial_nb             macro-F1 0.613 -> 0.680 after cutoff tuning  (39s)
complement_nb              macro-F1 0.647 -> 0.679 after cutoff tuning  (34s)
logreg_unweighted          macro-F1 0.639 -> 0.711 after cutoff tuning  (55s)
logreg_balanced            macro-F1 0.698 -> 0.706 after cutoff tuning  (60s)
sgd_hinge_balanced         macro-F1 0.702 -> 0.705 after cutoff tuning  (31s)
wordchar_logreg_balanced   macro-F1 0.697 -> 0.701 after cutoff tuning  (270s)
wordchar_svm_balanced      macro-F1 0.705 -> 0.701 after cutoff tuning  (162s)


In [3]:
pd.DataFrame(rows).set_index('model')[['macro_f1_untuned', 'macro_f1', 'accuracy', 'recall_not satisfied', 'recall_neutral', 'recall_satisfied']] \
  .sort_values('macro_f1', ascending=False).round(3)

,macro_f1_untuned,macro_f1,accuracy,recall_not satisfied,recall_neutral,recall_satisfied
model,,,,,,
logreg_unweighted,0.639,0.711,0.869,0.683,0.567,0.930
logreg_balanced,0.698,0.706,0.864,0.729,0.543,0.920
sgd_hinge_balanced,0.702,0.705,0.868,0.667,0.571,0.930
wordchar_svm_balanced,0.705,0.701,0.865,0.624,0.589,0.932
wordchar_logreg_balanced,0.697,0.701,0.865,0.679,0.535,0.929
multinomial_nb,0.613,0.680,0.863,0.683,0.430,0.937
complement_nb,0.647,0.679,0.859,0.660,0.472,0.931
